# Notebook 3: Causal Attribution of Shared Savings
## Difference-in-Differences + PSM + MSSP Projection

This notebook answers the core ACO analytics question:

> **"Did our intervention reduce total cost of care, and by how much does that translate to shared savings?"**

Methods:
1. **Difference-in-Differences (DiD)** — primary causal estimator
2. **Parallel trends validation** — key DiD assumption check
3. **Propensity Score Matching (PSM)** — sensitivity analysis
4. **Shared savings projection** — MSSP/ACO REACH benchmarking logic
5. **Heterogeneous treatment effects** — impact by risk tier


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.formula.api as smf
import warnings; warnings.filterwarnings("ignore")

from src.data_generator import generate_beneficiary_cohort, generate_utilization_panel
from src.causal_attribution import (
    difference_in_differences, propensity_score_matching,
    project_shared_savings, run_full_attribution
)

sns.set_style("whitegrid")
ACCENT = "#1B4F72"
TREAT_COLOR = ACCENT
CTRL_COLOR  = "#2E86C1"
print("Imports OK")


## 1. Generate Panel Data

In [ ]:
# True causal effect embedded in data: -$420/member
TRUE_EFFECT = -420.0

cohort = generate_beneficiary_cohort(n=50_000)
panel  = generate_utilization_panel(cohort, intervention_effect_pmpm=TRUE_EFFECT)
panel  = panel.merge(cohort[["bene_id","dual_eligible","risk_tier"]], on="bene_id", how="left")

print(f"Panel shape: {panel.shape}")
print(f"Beneficiaries: {panel['bene_id'].nunique():,}")
print(f"Intervention arm: {cohort['intervention'].sum():,} ({cohort['intervention'].mean():.1%})")
print(f"\nPre-period mean costs:")
pre = panel[panel["year"]==0]
print(pre.groupby("intervention")["total_cost"].mean().rename({0:"Control",1:"Treated"}).to_string())


## 2. Visual Pre/Post Comparison

In [ ]:
means = panel.groupby(["period","intervention"])["total_cost"].mean().reset_index()
means["group"] = means["intervention"].map({0:"Control",1:"Intervention"})
means["time"]  = means["period"].map({"pre":0,"post":1})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# DiD plot
for group, color in [("Control", CTRL_COLOR), ("Intervention", TREAT_COLOR)]:
    sub = means[means["group"]==group].sort_values("time")
    axes[0].plot([0,1], sub["total_cost"].values, "o-",
                 color=color, linewidth=2.5, markersize=9, label=group)
    for _, r in sub.iterrows():
        axes[0].annotate(f"${r['total_cost']:,.0f}",
                         xy=(r["time"], r["total_cost"]),
                         xytext=(8, 4), textcoords="offset points",
                         fontsize=9, color=color, fontweight="bold")

axes[0].set_xticks([0,1])
axes[0].set_xticklabels(["Pre-Intervention","Post-Intervention"], fontsize=11)
axes[0].set_ylabel("Mean Annual Cost per Beneficiary ($)", fontsize=11)
axes[0].set_title("Difference-in-Differences\nCost Trends by Arm", fontsize=12, fontweight="bold")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))
axes[0].legend(fontsize=10)

# Cost distributions post
post = panel[panel["year"]==1]
for intervention, color, label in [(1,TREAT_COLOR,"Intervention"),(0,CTRL_COLOR,"Control")]:
    sub = post[post["intervention"]==intervention]["total_cost"]
    axes[1].hist(sub, bins=60, alpha=0.5, color=color, label=f"{label} (mean=${sub.mean():,.0f})", density=True)
axes[1].set_xlabel("Annual Cost ($)"); axes[1].set_ylabel("Density")
axes[1].set_title("Post-Period Cost Distribution", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


## 3. Difference-in-Differences (Primary Estimator)

**Model:** `Cost_it = α + β₁·Post_t + β₂·Treat_i + β₃·(Post_t × Treat_i) + γ·X_i + ε_it`

β₃ is our ATT — the causal effect of the intervention.

In [ ]:
did = difference_in_differences(panel, "total_cost", ["age","dual_eligible"])

print("DIFFERENCE-IN-DIFFERENCES RESULTS")
print("="*50)
print(f"  ATT (cost/member):    ${did['att']:,.2f}")
print(f"  Standard error:       ${did['att_se']:,.2f}")
print(f"  95% CI:               [${did['ci_95_low']:,.2f},  ${did['ci_95_high']:,.2f}]")
print(f"  p-value:              {did['p_value']:.6f}")
print(f"  Statistically sig.:   {did['significant']}")
print(f"  N treated:            {did['n_treated']:,}")
print(f"  N control:            {did['n_control']:,}")
print(f"\nParallel Trends Check:")
print(f"  p-value (pre-trend test): {did['parallel_trends_p']:.3f}")
print(f"  {'✓ PASSES' if did['parallel_trends_p'] > 0.05 else '✗ FAILS'} (p > 0.05 required)")
print(f"\nTrue embedded effect:  ${TRUE_EFFECT:,.0f}/member")
print(f"Recovery accuracy:      {abs(did['att']-TRUE_EFFECT)/abs(TRUE_EFFECT)*100:.1f}% error")


## 4. PSM Sensitivity Analysis

DiD and PSM should converge. If they do, the finding is robust.

In [ ]:
psm = propensity_score_matching(panel, "total_cost")

print("PROPENSITY SCORE MATCHING RESULTS")
print("="*50)
print(f"  ATT (PSM):            ${psm['att']:,.2f}/member")
print(f"  95% CI:               [${psm['ci_95_low']:,.2f},  ${psm['ci_95_high']:,.2f}]")
print(f"  p-value:              {psm['p_value']:.6f}")
print(f"  Matched pairs:        {psm['n_matched_pairs']:,}")
print(f"  Post-match SMD (age): {psm['smd_age_post_match']:.3f}  (target < 0.10)")
print(f"  {'✓ BALANCED' if psm['smd_age_post_match'] < 0.10 else '✗ IMBALANCED'}")

print(f"\nCONVERGENCE CHECK:")
print(f"  DiD estimate:  ${did['att']:,.2f}")
print(f"  PSM estimate:  ${psm['att']:,.2f}")
diff = abs(did['att'] - psm['att'])
print(f"  Difference:    ${diff:.2f}  → {'✓ Robust finding' if diff < 50 else '⚠ Investigate'}")


## 5. Heterogeneous Treatment Effects by Risk Tier

Does the intervention work better for high-risk patients?

In [ ]:
hte_results = []
for tier in ["low", "moderate", "high"]:
    sub = panel[panel["risk_tier"] == tier]
    r = difference_in_differences(sub, "total_cost")
    hte_results.append({
        "Risk Tier": tier.capitalize(),
        "N (treated)": r["n_treated"],
        "ATT ($/member)": r["att"],
        "p-value": r["p_value"],
        "Significant": "✓" if r["significant"] else "✗"
    })

hte_df = pd.DataFrame(hte_results)
print("HETEROGENEOUS TREATMENT EFFECTS BY RISK TIER")
print(hte_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#85C1E9","#2E86C1",ACCENT]
bars = ax.bar(hte_df["Risk Tier"], hte_df["ATT ($/member)"],
              color=colors, edgecolor="white")
for bar, row in zip(bars, hte_results):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() - 15,
            f"${row['ATT ($/member)']:,.0f}\np={row['p-value']:.3f}",
            ha="center", va="top", fontsize=9, color="white", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("ATT — Cost Reduction per Member ($)", fontsize=11)
ax.set_title("Heterogeneous Treatment Effects by Risk Tier\nLarger savings in high-risk patients",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Shared Savings Projection (MSSP Framework)

In [ ]:
n_lives = cohort["intervention"].sum()
savings = project_shared_savings(
    att_pmpm=did["att"],
    n_attributed_lives=n_lives,
    benchmark_pmpm=9_800.0,
    mssp_sharing_rate=0.50,
    minimum_savings_rate=0.02,
)

print("MSSP SHARED SAVINGS PROJECTION")
print("="*50)
print(f"  Attributed lives:       {savings['n_attributed_lives']:,}")
print(f"  Benchmark PMPM:         ${savings['benchmark_pmpm']:,.2f}")
print(f"  Actual PMPM:            ${savings['actual_pmpm']:,.2f}")
print(f"  Gross savings total:    ${savings['gross_savings_total']:,.0f}")
print(f"  Savings rate:           {savings['savings_rate_pct']:.1f}%")
print(f"  Exceeds MSR (2%):       {'✓ YES' if savings['exceeds_msr'] else '✗ NO'}")
print(f"  Sharing rate:           {savings['mssp_sharing_rate']:.0%}")
print(f"  ─────────────────────────────────────────")
print(f"  SHARED SAVINGS EARNED:  ${savings['shared_savings_earned']:,.0f}")

# Sensitivity analysis across sharing rates
fig, ax = plt.subplots(figsize=(8, 4))
sharing_rates = np.arange(0.30, 0.81, 0.05)
earned = [project_shared_savings(did["att"], n_lives, sharing_rate=r)["shared_savings_earned"]
          for r in sharing_rates]
ax.plot(sharing_rates * 100, [e/1e6 for e in earned], "o-",
        color=ACCENT, linewidth=2.5, markersize=8)
ax.axvline(50, color="red", linestyle="--", linewidth=1.5, label="Standard MSSP (50%)")
ax.fill_between(sharing_rates * 100, [e/1e6 for e in earned], alpha=0.15, color=ACCENT)
ax.set_xlabel("MSSP Sharing Rate (%)", fontsize=11)
ax.set_ylabel("Shared Savings Earned ($M)", fontsize=11)
ax.set_title("Shared Savings Sensitivity to Sharing Rate", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:.1f}M"))
plt.tight_layout()
plt.show()
